# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## My prototype: the Spanish tutor from the week 1 exercise

In week 1 I built a Spanish tutor that generated flashcards for Flashcards.com. This is the same
idea as a real prototype, using everything from week 2:

| Week 2 skill | How it shows up here |
| --- | --- |
| Gradio UI | a `gr.Blocks` screen with a chat, a model switcher and a download |
| Streaming | replies appear token by token as they arrive |
| System prompt | turns the model into a tutor with rules, e.g. always give nouns their article |
| Switching models | one dropdown over GPT, Claude, Gemini and any local Ollama model you have |
| Tools | `conjugate_verb` and `save_flashcards` - real Python the model calls |
| Audio (the bold bonus) | speak your question into the microphone, and hear the answer back |

It runs with whatever you have: every provider with an API key in `.env` appears in the dropdown, and
so does every model that a local `ollama serve` is hosting. The same code is in
`spanish_tutor_gradio.py`, so it can also be run outside Jupyter with
`python week2/spanish_tutor_gradio.py`.

### Setup

In [10]:
import json
import os

import gradio as gr
import requests
from dotenv import load_dotenv
from openai import OpenAI

# The two tools live in tutor_tools.py - plain Python, no LLM involved
from tutor_tools import TOOL_FUNCTIONS, TOOLS, conjugate_verb, latest_deck

load_dotenv(override=True)

OLLAMA_URL = "http://localhost:11434/v1"

### The system prompt is where the expertise lives

This is what turns a general model into a tutor: the rules it must follow (articles on every noun),
and - importantly - when it must stop guessing and call a tool instead.

In [11]:
SYSTEM_PROMPT = """You are an expert Spanish tutor for an English-speaking learner.
Explain mistakes clearly and briefly, and keep every answer practical: a learner should be
able to use what you say in a conversation the same day.

Rules you always follow:
- Whenever you write a Spanish noun, include its definite article: el, la, los or las.
  Write "la manzana" and "el plátano", never "manzana" on its own.
- Mark the stressed syllable of a new word when the pronunciation is not obvious.
- When you are asked for a conjugation, call the conjugate_verb tool and report what it
  returns. Never write a conjugation table from memory.
- When the learner asks for flashcards or a deck, call the save_flashcards tool so they get
  a CSV they can import. Write the cards yourself, then pass them to the tool.
- Answer in English unless the learner writes in Spanish or asks for Spanish.
Respond in markdown, without code blocks."""


WELCOME = (
    "¡Hola! I'm your Spanish tutor. Ask me anything about Spanish - or try "
    "*\"conjugate tener in the future\"* or *\"make me a deck of 10 kitchen words\"*."
)

### Which models can we talk to?

Each provider that has a key becomes an entry in the dropdown. Anthropic and Google both offer an
OpenAI-compatible endpoint, so one client library covers everything, and Ollama is asked directly
which models it is hosting.

In [12]:
def ollama_models():
    """Ask a local Ollama server which models it has; an empty list if it is not running."""
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        response.raise_for_status()
        return [model["name"] for model in response.json().get("models", [])]
    except Exception:
        return []


def available_models():
    """Map a friendly label to (client, model) for every provider that is configured."""
    models = {}

    if os.getenv("OPENAI_API_KEY"):
        client = OpenAI()
        models["GPT-4.1-mini"] = (client, "gpt-4.1-mini")
    if os.getenv("ANTHROPIC_API_KEY"):
        client = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url=ANTHROPIC_URL)
        models["Claude Haiku 4.5"] = (client, "claude-haiku-4-5")
    if os.getenv("GOOGLE_API_KEY"):
        client = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=GEMINI_URL)
        models["Gemini 2.5 Flash"] = (client, "gemini-2.5-flash")

    ollama = OpenAI(base_url=OLLAMA_URL, api_key="ollama")
    for name in ollama_models():
        models[f"Ollama: {name}"] = (ollama, name)

    return models


MODELS = available_models()

print("Models in the dropdown:", list(MODELS) or "none - add an API key or run `ollama serve`")

Models in the dropdown: ['Ollama: llama3.2:latest', 'Ollama: qwen2.5:14b']


### The tools

A language model is a poor conjugator: it recites forms it has seen rather than applying the rule,
and it cannot write a file at all. So two plain Python functions in `tutor_tools.py` do that work,
and the system prompt tells the model to call them.

In [13]:
# A quick look at the tools before we hand them to a model

print(conjugate_verb("tener", "future"))
print()
print("Tools offered to the model:", [tool["function"]["name"] for tool in TOOLS])

TOOL CALLED: conjugate_verb('tener', 'future')
tener - future (irregular stem)
yo: tendré
tú: tendrás
él/ella/usted: tendrá
nosotros: tendremos
vosotros: tendréis
ellos/ellas/ustedes: tendrán

Tools offered to the model: ['conjugate_verb', 'save_flashcards']


### Streaming, with tool calls along the way

Streaming and tools together need a little care: tool calls arrive in fragments, exactly like text
does, so we reassemble them. If the model finished by asking for tools, we run them, add the results
to the conversation and stream again - it usually answers properly the second time round.

In [14]:
def collect_tool_calls(chunks_of_tool_calls, accumulated):
    """Rebuild whole tool calls from the fragments that arrive in a stream."""
    for delta in chunks_of_tool_calls:
        call = accumulated.setdefault(
            delta.index, {"id": "", "type": "function", "function": {"name": "", "arguments": ""}}
        )
        if delta.id:
            call["id"] = delta.id
        if delta.function and delta.function.name:
            call["function"]["name"] = delta.function.name
        if delta.function and delta.function.arguments:
            call["function"]["arguments"] += delta.function.arguments
    return accumulated


def run_tool_calls(tool_calls, decks):
    """Run each tool the model asked for and build the tool messages that answer it."""
    results = []
    for call in tool_calls:
        name = call["function"]["name"]
        function = TOOL_FUNCTIONS.get(name)
        if function is None:
            content = f"There is no tool called {name}."
        else:
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
                content = function(**arguments)
                if name == "save_flashcards" and latest_deck():
                    decks.append(latest_deck())
            except Exception as error:
                content = f"The {name} tool failed: {type(error).__name__}: {error}"
        results.append({"role": "tool", "content": content, "tool_call_id": call["id"]})
    return results


def stream_reply(messages, client, model, decks):
    """Stream the assistant's reply, pausing to run any tools it asks for along the way."""
    while True:
        stream = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, stream=True
        )
        reply = ""
        tool_calls = {}
        for chunk in stream:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            if delta.content:
                reply += delta.content
                yield reply
            if delta.tool_calls:
                collect_tool_calls(delta.tool_calls, tool_calls)

        if not tool_calls:
            return

        calls = [tool_calls[index] for index in sorted(tool_calls)]
        # Some providers reject a null content, so only include it when the model said something
        asking_for_tools = {"role": "assistant", "tool_calls": calls}
        if reply:
            asking_for_tools["content"] = reply
        messages.append(asking_for_tools)
        messages.extend(run_tool_calls(calls, decks))

### Listening and speaking

The bold bonus. Both use OpenAI's audio models, so they announce themselves politely when there is
no OpenAI key rather than breaking the app.

In [15]:
def transcribe(audio_path):
    """Turn a recording into text with OpenAI's speech-to-text model."""
    if not os.getenv("OPENAI_API_KEY"):
        raise gr.Error("Voice input needs an OPENAI_API_KEY in your .env - type your message instead.")
    with open(audio_path, "rb") as audio_file:
        transcription = OpenAI().audio.transcriptions.create(
            model="gpt-4o-mini-transcribe", file=audio_file
        )
    return transcription.text


def speak(text):
    """Read the reply out loud; returns None when text-to-speech is not available."""
    if not os.getenv("OPENAI_API_KEY") or not text.strip():
        return None
    try:
        response = OpenAI().audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input=text,
            instructions="Speak warmly, like a patient language teacher. Pronounce Spanish words as a native speaker would.",
        )
        return response.content
    except Exception as error:
        gr.Warning(f"Could not generate audio: {type(error).__name__}")
        return None

### The callback Gradio will call

It yields four things every time: the (now empty) textbox, the conversation, any audio, and the
flashcard deck to download. The first `yield` clears the box immediately so the UI feels quick.

In [16]:
def chat(message, history, model_label, voice_on):
    """The Gradio callback: stream a reply, then optionally speak it and offer any new deck."""
    if not MODELS:
        raise gr.Error(
            "No models available. Add an API key to .env, or start a local model with `ollama serve`."
        )
    if not message.strip():
        return

    client, model = MODELS[model_label]
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [
        {"role": "user", "content": message}
    ]
    chat_history = history + [{"role": "user", "content": message}]
    decks = []

    yield "", chat_history + [{"role": "assistant", "content": ""}], None, gr.update()

    reply = ""
    for reply in stream_reply(messages, client, model, decks):
        yield "", chat_history + [{"role": "assistant", "content": reply}], None, gr.update()

    chat_history += [{"role": "assistant", "content": reply}]
    deck_update = gr.update(value=str(decks[-1]), visible=True) if decks else gr.update()
    yield "", chat_history, speak(reply) if voice_on else None, deck_update

### The UI

In [17]:
def build_ui():
    labels = list(MODELS)
    with gr.Blocks(title="Spanish Tutor", theme=gr.themes.Soft()) as ui:
        gr.Markdown("# 🇪🇸 Spanish Tutor\nStreaming, switchable models, real tools, and a voice.")
        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(
                    height=460,
                    type="messages",
                    value=[{"role": "assistant", "content": WELCOME}],
                    label="Tutor",
                )
            with gr.Column(scale=1):
                model_selector = gr.Dropdown(
                    labels, label="Model", value=labels[0] if labels else None
                )
                voice_on = gr.Checkbox(label="Speak the replies", value=False)
                audio_output = gr.Audio(label="Tutor's voice", autoplay=True)
                deck_file = gr.File(label="Your flashcard deck", visible=False)
                clear = gr.Button("Clear conversation")
        with gr.Row():
            message = gr.Textbox(
                label="Your message", placeholder="¿Cómo se dice...?", submit_btn=True, scale=4
            )
            microphone = gr.Audio(sources=["microphone"], type="filepath", label="Or speak", scale=1)

        inputs = [message, chatbot, model_selector, voice_on]
        outputs = [message, chatbot, audio_output, deck_file]
        message.submit(chat, inputs, outputs)
        microphone.stop_recording(transcribe, microphone, message).then(
            lambda: None, None, microphone
        ).then(chat, inputs, outputs)
        clear.click(
            lambda: ([{"role": "assistant", "content": WELCOME}], None, gr.update(visible=False)),
            None,
            [chatbot, audio_output, deck_file],
        )
    return ui

In [18]:
ui = build_ui()
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TOOL CALLED: conjugate_verb('hablar', ' futuro')


Traceback (most recent call last):
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/eugene_dyl/PycharmProjects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

### Things worth trying in the running app

- *"Conjugate poder in the preterite"* - watch `TOOL CALLED` appear here in Jupyter as the tutor
  uses the tool instead of its memory.
- *"Make me a deck of 12 words for ordering in a restaurant"* - the deck appears as a CSV download.
- Ask the same question of GPT and of a local Ollama model, and compare how well each one follows
  the rule about definite articles.
- Tick **Speak the replies**, and use the microphone to ask a question in Spanish.

To stop the server, run `ui.close()` in a new cell, or restart the kernel.